### Tyring to implement chromaDB on a large netflix movies dataset


### Step:1 Load the dataset

In [ ]:
import pandas as pd

df = pd.read_csv("netflix_titles.csv")

print(df.columns)
print(df.shape)
df.head()



Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='object')
(8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       2000 non-null   object
 1   type          2000 non-null   object
 2   title         2000 non-null   object
 3   director      1348 non-null   object
 4   cast          1800 non-null   object
 5   country       1610 non-null   object
 6   date_added    2000 non-null   object
 7   release_year  2000 non-null   int64 
 8   rating        2000 non-null   object
 9   duration      2000 non-null   object
 10  listed_in     2000 non-null   object
 11  description   2000 non-null   object
dtypes: int64(1), object(11)
memory usage: 187.6+ KB


### Step:2 Cleaning the dataset

In [ ]:
df = df.dropna(subset=['description'])
df = df.drop_duplicates(subset=['title']) 
df = df.reset_index(drop=True)

# Optional: cap dataset size while testing
#df = df.head(2000)  # remove/increase once it works end-to-end

### Step:3 Build the ChromaDB Collection

In [5]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="./chroma_db")

embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.get_or_create_collection(
    name="netflix_movies",
    embedding_function=embed_fn
)

### Step:4 Insert in batches (important for bigger datasets)

### Chroma can Choke if you try to add tens of thousands of rows in one call, so better to batch it.

In [11]:
batch_size = 100

for start in range(0, len(df), batch_size):
    end = start + batch_size
    batch = df.iloc[start:end]

    documents = batch["description"].astype(str).tolist()
    ids = batch["show_id"].astype(str).tolist()

    metadatas = []

    for _, row in batch.iterrows():
        metadatas.append({
            "title": row["title"],
            "release_year": int(row["release_year"]),
            "type": row["type"],
            "director": "" if pd.isna(row["director"]) else row["director"],
            "cast": "" if pd.isna(row["cast"]) else row["cast"],
            "country": "" if pd.isna(row["country"]) else row["country"],
            "rating": row["rating"],
            "duration": row["duration"],
            "listed_in": row["listed_in"]
        })

    collection.add(
        documents=documents,
        ids=ids,
        metadatas=metadatas
    )

    print(f"Inserted rows {start}-{min(end, len(df))}")

print("Total in collection:", collection.count())

Inserted rows 0-100
Inserted rows 100-200
Inserted rows 200-300
Inserted rows 300-400
Inserted rows 400-500
Inserted rows 500-600
Inserted rows 600-700
Inserted rows 700-800
Inserted rows 800-900
Inserted rows 900-1000
Inserted rows 1000-1100
Inserted rows 1100-1200
Inserted rows 1200-1300
Inserted rows 1300-1400
Inserted rows 1400-1500
Inserted rows 1500-1600
Inserted rows 1600-1700
Inserted rows 1700-1800
Inserted rows 1800-1900
Inserted rows 1900-2000
Total in collection: 2000


### Step:5. Simple similarity search function

In [12]:
def search_movies(query, n_results=5):
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    for i, result in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]

        print(f"Result {i + 1}:")
        print(f"Title: {metadata['title']}")
        print(f"Description: {result}")
        print(f"Release Year: {metadata['release_year']}")
        print(f"Type: {metadata['type']}")
        print(f"Director: {metadata['director']}")
        print(f"Cast: {metadata['cast']}")
        print(f"Country: {metadata['country']}")
        print(f"Rating: {metadata['rating']}")
        print(f"Duration: {metadata['duration']}")
        print(f"Listed In: {metadata['listed_in']}")
        print("-" * 50)

In [16]:
# Try it
search_movies("Best indian movies")

Result 1:
Title: Shootout at Lokhandwala
Description: Based on a true story, this action film follows an incident that stunned a nation in the early 1990s. In Mumbai, India, the notorious gangster Maya holds off veteran cop Khan and a force of more than 200 policemen in a six-hour bloody gunfight.
Release Year: 2007
Type: Movie
Director: Apoorva Lakhia
Cast: Amitabh Bachchan, Sanjay Dutt, Sunil Shetty, Arbaaz Khan, Abhishek Bachchan, Vivek Oberoi, Tusshar Kapoor, Rohit Roy, Shabbir Ahluwalia, Dia Mirza, Amrita Singh, Neha Dhupia
Country: India
Rating: TV-MA
Duration: 116 min
Listed In: Action & Adventure, Dramas, International Movies
--------------------------------------------------
Result 2:
Title: Madness in the Desert
Description: The story of making "Lagaan," one of the millennium's seminal Indian films, is told from the point of view of production team member Satyajit Bhatkal.
Release Year: 2004
Type: Movie
Director: Satyajit Bhatkal
Cast: Aamir Khan, Ashutosh Gowariker
Country: 